In [ ]:
# Reload, process, and clean the data in one step
import pandas as pd
import numpy as np
from sklearn.ensemble import RandomForestRegressor
from sklearn.model_selection import train_test_split
from sklearn.metrics import mean_squared_error
import pickle

# Load and parse data
trait_prefixes = ['EXT', 'EST', 'AGR', 'CSN', 'OPN']
df = pd.read_csv('data-final.csv', sep='\t', low_memory=False)
for trait in trait_prefixes:
    cols = [f'{trait}{i}' for i in range(1, 11)]
    df[f'{trait}_score'] = df[cols].mean(axis=1)
question_cols = [f'{trait}{i}' for trait in trait_prefixes for i in range(1,11)]
score_cols = [f'{trait}_score' for trait in trait_prefixes]
df_clean = df[question_cols + score_cols].dropna(subset=score_cols)
print('Rows after cleaning:', len(df_clean))
print('NaNs in trait scores:', df_clean[score_cols].isnull().sum())

# Train model
X = df_clean[question_cols]
y = df_clean[score_cols]
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)
model = RandomForestRegressor(n_estimators=100, random_state=42, n_jobs=-1)
model.fit(X_train, y_train)
y_pred = model.predict(X_test)
mse = mean_squared_error(y_test, y_pred)
print('Test MSE:', mse)
with open('travel_personality_model.pkl', 'wb') as f:
    pickle.dump(model, f)
print('Model saved as travel_personality_model.pkl')

In [ ]:
# Install scikit-learn for model training
%pip install scikit-learn

In [ ]:
# Install required packages
%pip install pandas numpy

# Big Five Personality Data Preparation
This notebook loads, inspects, and processes the Big Five survey data for ML model training.

In [ ]:
import pandas as pd
import numpy as np
# Load the dataset
df = pd.read_csv('data-final.csv', low_memory=False)
print('Shape:', df.shape)
df.head()

In [ ]:
# Reload the dataset with tab delimiter
# This will split the single column into proper columns

df = pd.read_csv('data-final.csv', sep='\t', low_memory=False)
print('Shape:', df.shape)
print('Columns:', df.columns.tolist())
df.head()

In [ ]:
# Compute Big Five Trait Scores
trait_prefixes = ['EXT', 'EST', 'AGR', 'CSN', 'OPN']
for trait in trait_prefixes:
    cols = [f'{trait}{i}' for i in range(1, 11)]
    df[f'{trait}_score'] = df[cols].mean(axis=1)
df[['EXT_score', 'EST_score', 'AGR_score', 'CSN_score', 'OPN_score']].head()

In [ ]:
# Use cleaned data for training
X = df_clean[question_cols]
y = df_clean[score_cols]

X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

model = RandomForestRegressor(n_estimators=100, random_state=42, n_jobs=-1)
model.fit(X_train, y_train)

# Evaluate model
y_pred = model.predict(X_test)
mse = mean_squared_error(y_test, y_pred)
print('Test MSE:', mse)

# Save model
import pickle
with open('travel_personality_model.pkl', 'wb') as f:
    pickle.dump(model, f)
print('Model saved as travel_personality_model.pkl')

In [ ]:
# Remove rows with NaN values in question or score columns
df_clean = df_clean.dropna(subset=question_cols + score_cols)
print('Rows after dropping NaNs:', len(df_clean))

In [ ]:
# Check for NaNs in target columns before training
print('NaNs in trait scores:')
print(df_clean[score_cols].isnull().sum())

In [ ]:
# Drop rows with NaNs in trait score columns only
df_clean = df_clean.dropna(subset=score_cols)
print('Rows after dropping NaNs in trait scores:', len(df_clean))
print('NaNs in trait scores after cleaning:')
print(df_clean[score_cols].isnull().sum())

In [ ]:
# Display all column names and a sample of the data
print('Columns:', df.columns.tolist())
df.head()

## Compute Big Five Trait Scores
Each trait score is the average of its 10 questions.

In [ ]:
trait_prefixes = ['EXT', 'EST', 'AGR', 'CSN', 'OPN']
for trait in trait_prefixes:
    cols = [f'{trait}{i}' for i in range(1, 11)]
    df[f'{trait}_score'] = df[cols].mean(axis=1)
df[['EXT_score', 'EST_score', 'AGR_score', 'CSN_score', 'OPN_score']].head()

## Drop Unnecessary Columns
Keep only trait scores and any columns you want for ML.

In [ ]:
keep_cols = ['EXT_score', 'EST_score', 'AGR_score', 'CSN_score', 'OPN_score']
df_clean = df[keep_cols]
df_clean.head()

## Save Processed Dataset
This file is ready for ML model training.

In [ ]:
df_clean.to_csv('bigfive_processed.csv', index=False)
print('Saved processed dataset as bigfive_processed.csv')

In [ ]:
# Generate confusion matrix and ROC curve for EXT_score classification
import pandas as pd
import numpy as np
from sklearn.ensemble import RandomForestClassifier
from sklearn.model_selection import train_test_split
from sklearn.metrics import confusion_matrix, ConfusionMatrixDisplay, roc_curve, auc
import matplotlib.pyplot as plt

# Load and clean data
trait_prefixes = ['EXT', 'EST', 'AGR', 'CSN', 'OPN']
df = pd.read_csv('data-final.csv', sep='\t', low_memory=False)
for trait in trait_prefixes:
    cols = [f'{trait}{i}' for i in range(1, 11)]
    df[f'{trait}_score'] = df[cols].mean(axis=1)
question_cols = [f'{trait}{i}' for trait in trait_prefixes for i in range(1,11)]
score_cols = [f'{trait}_score' for trait in trait_prefixes]
df_clean = df[question_cols + score_cols].dropna(subset=score_cols)

# Binarize EXT_score for classification (e.g., EXT_score > 3 is 'high', else 'low')
df_clean['EXT_high'] = (df_clean['EXT_score'] > 3).astype(int)

# Train/test split
X = df_clean[question_cols]
y = df_clean['EXT_high']
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

# Train classifier
clf = RandomForestClassifier(n_estimators=100, random_state=42)
clf.fit(X_train, y_train)
y_pred = clf.predict(X_test)
y_proba = clf.predict_proba(X_test)[:, 1]

# Confusion Matrix
cm = confusion_matrix(y_test, y_pred)
disp = ConfusionMatrixDisplay(confusion_matrix=cm)
disp.plot()
plt.title('Confusion Matrix for EXT_score (High vs Low)')
plt.show()

# ROC Curve
fpr, tpr, _ = roc_curve(y_test, y_proba)
roc_auc = auc(fpr, tpr)
plt.figure()
plt.plot(fpr, tpr, color='darkorange', lw=2, label='ROC curve (area = %0.2f)' % roc_auc)
plt.plot([0, 1], [0, 1], color='navy', lw=2, linestyle='--')
plt.xlim([0.0, 1.0])
plt.ylim([0.0, 1.05])
plt.xlabel('False Positive Rate')
plt.ylabel('True Positive Rate')
plt.title('Receiver Operating Characteristic (EXT_score High)')
plt.legend(loc="lower right")
plt.show()

In [ ]:
# Load a smaller sample to avoid memory errors and generate confusion matrix and ROC curve
import pandas as pd
import numpy as np
from sklearn.ensemble import RandomForestClassifier
from sklearn.model_selection import train_test_split
from sklearn.metrics import confusion_matrix, ConfusionMatrixDisplay, roc_curve, auc
import matplotlib.pyplot as plt

# Try loading only 10,000 rows
trait_prefixes = ['EXT', 'EST', 'AGR', 'CSN', 'OPN']
df = pd.read_csv('data-final.csv', sep='\t', low_memory=False, nrows=10000)
for trait in trait_prefixes:
    cols = [f'{trait}{i}' for i in range(1, 11)]
    df[f'{trait}_score'] = df[cols].mean(axis=1)
question_cols = [f'{trait}{i}' for trait in trait_prefixes for i in range(1,11)]
score_cols = [f'{trait}_score' for trait in trait_prefixes]
df_clean = df[question_cols + score_cols].dropna(subset=score_cols)

# Binarize EXT_score for classification (e.g., EXT_score > 3 is 'high', else 'low')
df_clean['EXT_high'] = (df_clean['EXT_score'] > 3).astype(int)

# Train/test split
X = df_clean[question_cols]
y = df_clean['EXT_high']
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

# Train classifier
clf = RandomForestClassifier(n_estimators=100, random_state=42)
clf.fit(X_train, y_train)
y_pred = clf.predict(X_test)
y_proba = clf.predict_proba(X_test)[:, 1]

# Confusion Matrix
cm = confusion_matrix(y_test, y_pred)
disp = ConfusionMatrixDisplay(confusion_matrix=cm)
disp.plot()
plt.title('Confusion Matrix for EXT_score (High vs Low)')
plt.show()

# ROC Curve
fpr, tpr, _ = roc_curve(y_test, y_proba)
roc_auc = auc(fpr, tpr)
plt.figure()
plt.plot(fpr, tpr, color='darkorange', lw=2, label='ROC curve (area = %0.2f)' % roc_auc)
plt.plot([0, 1], [0, 1], color='navy', lw=2, linestyle='--')
plt.xlim([0.0, 1.0])
plt.ylim([0.0, 1.05])
plt.xlabel('False Positive Rate')
plt.ylabel('True Positive Rate')
plt.title('Receiver Operating Characteristic (EXT_score High)')
plt.legend(loc="lower right")
plt.show()

In [ ]:
# Trait Score Distribution Plots
import matplotlib.pyplot as plt
trait_prefixes = ['EXT', 'EST', 'AGR', 'CSN', 'OPN']
for trait in trait_prefixes:
    plt.figure()
    plt.hist(df_clean[f'{trait}_score'], bins=30, alpha=0.7, color='skyblue')
    plt.title(f'Distribution of {trait}_score')
    plt.xlabel(f'{trait}_score')
    plt.ylabel('Frequency')
    plt.tight_layout()
    plt.show()
# Feature Importance Bar Chart
importances = clf.feature_importances_
indices = np.argsort(importances)[::-1]
feature_names = X_test.columns
plt.figure(figsize=(10, 6))
plt.title('Feature Importances')
plt.bar(range(len(importances)), importances[indices], align='center')
plt.xticks(range(len(importances)), [feature_names[i] for i in indices], rotation=90)
plt.tight_layout()
plt.show()

In [ ]:
# Load a smaller sample and rerun visualizations
import pandas as pd
import numpy as np
from sklearn.ensemble import RandomForestClassifier
from sklearn.model_selection import train_test_split
import matplotlib.pyplot as plt

# Load only 10,000 rows
trait_prefixes = ['EXT', 'EST', 'AGR', 'CSN', 'OPN']
df = pd.read_csv('data-final.csv', sep='\t', low_memory=False, nrows=10000)
for trait in trait_prefixes:
    cols = [f'{trait}{i}' for i in range(1, 11)]
    df[f'{trait}_score'] = df[cols].mean(axis=1)
question_cols = [f'{trait}{i}' for trait in trait_prefixes for i in range(1,11)]
score_cols = [f'{trait}_score' for trait in trait_prefixes]
df_clean = df[question_cols + score_cols].dropna(subset=score_cols)

# Binarize EXT_score for classification
if 'EXT_high' not in df_clean.columns:
    df_clean['EXT_high'] = (df_clean['EXT_score'] > 3).astype(int)

# Train/test split
X = df_clean[question_cols]
y = df_clean['EXT_high']
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

# Train classifier
clf = RandomForestClassifier(n_estimators=50, random_state=42)
clf.fit(X_train, y_train)

# Trait Score Distribution Plots
for trait in trait_prefixes:
    plt.figure()
    plt.hist(df_clean[f'{trait}_score'], bins=30, alpha=0.7, color='skyblue')
    plt.title(f'Distribution of {trait}_score')
    plt.xlabel(f'{trait}_score')
    plt.ylabel('Frequency')
    plt.tight_layout()
    plt.show()

# Feature Importance Bar Chart
importances = clf.feature_importances_
indices = np.argsort(importances)[::-1]
feature_names = X_test.columns
plt.figure(figsize=(10, 6))
plt.title('Feature Importances')
plt.bar(range(len(importances)), importances[indices], align='center')
plt.xticks(range(len(importances)), [feature_names[i] for i in indices], rotation=90)
plt.tight_layout()
plt.show()